In [ ]:
!git clone https://github.com/Thom-320/scres-ia.git /content/scresia


Cloning into '/content/scresia'...
remote: Enumerating objects: 1438, done.
remote: Counting objects: 100% (1438/1438), done.
remote: Compressing objects: 100% (680/680), done.
remote: Total 1438 (delta 896), reused 1281 (delta 740), pack-reused 0 (from 0)
Receiving objects: 100% (1438/1438), 5.60 MiB | 13.62 MiB/s, done.
Resolving deltas: 100% (896/896), done.


In [1]:
!pip install simpy

In [2]:
!pip install stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 4.8 MB/s eta 0:00:0000:01


In [3]:
from stable_baselines3.common.monitor import Monitor

2026-05-18 01:31:19.376533: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779067879.576131      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779067879.634279      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779067880.101472      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779067880.101512      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779067880.101515      58 computation_placer.cc:177] computation placer alr

In [5]:
import sys
sys.path.insert(0, "/content")

#from scresia.supply_chain import MFSCSimulation
#from scresia.supply_chain.config import SIMULATION_HORIZON
from scresia.supply_chain.external_env_interface import  make_dkana_thesis_faithful_env

In [ ]:
import shutil, sys

shutil.rmtree("/content/scresia", ignore_errors=True)

In [4]:
!git clone https://github.com/Thom-320/scres-ia.git /content/scresia

Cloning into '/content/scresia'...
remote: Enumerating objects: 1438, done.
remote: Counting objects: 100% (1438/1438), done.
remote: Compressing objects: 100% (680/680), done.
remote: Total 1438 (delta 896), reused 1281 (delta 740), pack-reused 0 (from 0)
Receiving objects: 100% (1438/1438), 5.60 MiB | 21.24 MiB/s, done.
Resolving deltas: 100% (896/896), done.


In [7]:
import shimmy

In [9]:

from stable_baselines3.common.vec_env import VecFrameStack, SubprocVecEnv
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3 import PPO
import gymnasium as gym

venv=DummyVecEnv([lambda:Monitor(make_dkana_thesis_faithful_env(
    reward_mode="control_v1",
    risk_level="increased",
    stochastic_pt=True,
    max_steps=260*8,
    step_size_hours=168
)) for _ in range(1)])
env=VecFrameStack(venv,n_stack=30)
from stable_baselines3.common.vec_env import VecNormalize



import torch
import numpy as np


In [16]:
#env.reset()
env.step([[0]*18])


RuntimeError: Tried to step environment that needs reset

In [10]:
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from einops import rearrange,repeat



class DMLPA(BaseFeaturesExtractor):
  def __init__(self,observation_space,factor,features_dim=120):
    super().__init__(observation_space,features_dim)
    self.obs_dimension=observation_space.shape[0]//factor
    #print(self.obs_dimension)

    self.latent_rw=torch.nn.Sequential(
        torch.nn.Linear(self.obs_dimension,100),
        torch.nn.GELU(),
        torch.nn.Linear(100,features_dim))

    self.attlayer=torch.nn.TransformerEncoderLayer(d_model=features_dim,nhead=12,batch_first=True)
    self.accumulated=torch.nn.TransformerEncoder(self.attlayer,num_layers=4)
  def forward(self,observations):
    batch_size=observations.shape[0]
    observations=rearrange(observations,"b (d k) -> b d k",k=self.obs_dimension)
    #print(observations.shape)

    observations=self.latent_rw(observations)
    observations=self.accumulated(observations)
    observations=observations[:,-1,:]
    #print(observations.shape)
    return observations



In [ ]:
env=VecFrameStack(env,n_stack=10)

In [19]:
model=DMLPA(env.observation_space,30,features_dim=120)

In [18]:
#get number of parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [19]:
count_parameters(model)

2775360

In [ ]:
test=torch.randn((4,48*2))

In [ ]:
model.forward(test).shape

torch.Size([4, 2, 24, 2])
torch.Size([4, 2, 24, 1])


torch.Size([4, 120])

In [ ]:
env.reset()

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.  

In [ ]:
test=torch.randn((1,5,10))

In [ ]:
test

tensor([[[ 1.1683,  0.2388, -0.9738,  0.3967, -0.2400,  0.4615, -1.2020,
          -0.6069,  0.2994,  0.6496],
         [ 0.5266,  0.4704, -0.4953,  0.4332,  0.0341, -0.3120,  0.8392,
          -0.8860, -0.3725,  0.4145],
         [-0.1490, -0.5811,  1.1343,  1.2282,  1.0776,  0.7445,  1.1265,
          -0.5841, -1.2635, -0.2759],
         [ 0.4043,  0.5532,  1.2528,  0.2920,  0.1463, -0.7386,  1.6762,
           0.1300, -0.0117, -0.5518],
         [ 0.5706,  0.4418, -1.1546, -0.0887,  0.1703,  0.0070, -1.1423,
          -0.7362, -3.2380, -0.2108]]])

In [ ]:
test=rearrange(test,'b h  (c k) -> b h c k',c=2)
test=rearrange(test,"b h c k -> b h k c")


In [ ]:
test.shape

torch.Size([1, 5, 5, 2])

In [ ]:
observation=env.step(torch.tensor([[0.1,0.1,0.1,0.1,0.1,0.1,0.1]]))

In [18]:


policy_kwargs = dict(
    features_extractor_class=DMLPA,
    features_extractor_kwargs=dict(features_dim=120,factor=30),
)

model = PPO(
    "MlpPolicy",
    env,
    policy_kwargs=policy_kwargs,
    device="cuda",   # optional
    verbose=1
)

Using cuda device


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [19]:
model.learn(total_timesteps=100000,reset_num_timesteps=False)

Process ForkServerProcess-16:
Process ForkServerProcess-15:
Process ForkServerProcess-14:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/subproc_vec_env.py", line 33, in _worker
    cmd, data = remote.recv()
                ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/subproc_vec_env.py", line 33, in _worker
    cmd, data = remote.recv()
                ^^^^^^^^^^^^^
  File "/

KeyboardInterrupt: 

In [ ]:
#verificar a las 8:30 si el modelo converge en cartpole
env.step([[1]*18])

(array([[0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e